In [ ]:
import os
import zipfile
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoConfig, AutoModel, DataCollatorWithPadding
from google.colab import files, drive

# ==========================================
# 1. UPLOAD VALID / TEST FILES VIA .upload()
# ==========================================
print("Click 'Choose Files' below to upload val.jsonl and/or test.jsonl (or a .zip):")
uploaded = files.upload()

os.makedirs("./demo_data", exist_ok=True)
for fname in uploaded.keys():
    if fname.lower().endswith(".zip") and zipfile.is_zipfile(fname):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall("./demo_data")
    else:
        os.replace(fname, os.path.join("./demo_data", fname))

# Locate uploaded val.jsonl and test.jsonl automatically
valid_path, test_path = None, None
for root, _, fnames in os.walk("./demo_data"):
    for f in fnames:
        if f in ["val.jsonl", "valid.jsonl"]:
            valid_path = os.path.join(root, f)
        elif f == "test.jsonl":
            test_path = os.path.join(root, f)

# ==========================================
# 2. CONNECT GOOGLE DRIVE & SHOW SAVED VALIDATION SCORES FIRST
# ==========================================
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/CS6315E_RTE_Submission"

variants = [
    ("Model 1 (seed=42, lr=2.0e-5)",   "submission_base_seed42_lr2e5"),
    ("Model 2 (seed=100, lr=1.5e-5)",  "submission_base_seed100_lr15e5"),
    ("Model 3 (seed=2026, lr=2.5e-5)", "submission_base_seed2026_lr25e5"),
]

label_map = {"not_entailment": 0, "entailment": 1}
id2label = {0: "not_entailment", 1: "entailment"}

if valid_path and os.path.exists(valid_path):
    val_df = pd.read_json(valid_path, lines=True)
    y_val_true = val_df["label"].map(label_map).values

    val_summary = []
    for display_name, folder_name in variants:
        val_pred_path = os.path.join(DRIVE_DIR, folder_name, "valid_predictions.jsonl")
        if os.path.exists(val_pred_path):
            vp_df = pd.read_json(val_pred_path, lines=True)
            y_val_pred = vp_df["label"].map(label_map).values
            val_summary.append({
                "Model Variant": display_name,
                "Saved Folder": folder_name,
                "Validation Accuracy": round(accuracy_score(y_val_true, y_val_pred), 4),
                "Validation F1": round(f1_score(y_val_true, y_val_pred, average="binary"), 4),
            })

    print("\n=================================================================")
    print("STEP 1: SAVED VALIDATION RESULTS (BEFORE RUNNING TEST DATA)")
    print("=================================================================")
    print(pd.DataFrame(val_summary).to_string(index=False))
else:
    print("\nNote: val.jsonl was not uploaded, skipping validation table.")

# ==========================================
# 3. DEFINE ARCHITECTURE & RUN ON TEST.JSONL (IF UPLOADED)
# ==========================================
class DebertaV3EntailmentMSD(nn.Module):
    def __init__(self, model_name_or_dir, num_labels=2, offline=False):
        super().__init__()
        if offline and os.path.exists(model_name_or_dir):
            self.config = AutoConfig.from_pretrained(model_name_or_dir, local_files_only=True)
            self.deberta = AutoModel.from_config(self.config, attn_implementation="eager")
        else:
            self.config = AutoConfig.from_pretrained(model_name_or_dir)
            self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
            self.deberta = AutoModel.from_pretrained(
                model_name_or_dir, config=self.config, attn_implementation="eager", dtype=torch.float32
            )
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        pooled = torch.sum(last_hidden * mask, 1) / mask.sum(1).clamp(min=1e-9)
        return sum([self.classifier(drop(pooled)) for drop in self.dropouts]) / len(self.dropouts)

if test_path and os.path.exists(test_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok_dir = os.path.join(DRIVE_DIR, "saved_tokenizer")
    cfg_dir = os.path.join(DRIVE_DIR, "saved_config")
    use_offline = os.path.exists(tok_dir) and os.path.exists(cfg_dir)

    if use_offline:
        tokenizer = AutoTokenizer.from_pretrained(tok_dir, use_fast=False, local_files_only=True)
        model_source = cfg_dir
    else:
        tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=False)
        model_source = "microsoft/deberta-v3-base"

    test_df = pd.read_json(test_path, lines=True)
    has_test_labels = "label" in test_df.columns and test_df["label"].notna().all()
    if has_test_labels:
        y_test_true = test_df["label"].map(label_map).values
        y_test_tensor = torch.tensor(y_test_true, dtype=torch.long)

    test_ds = Dataset.from_pandas(test_df[["premise", "hypothesis"]]).map(
        lambda b: tokenizer(b["premise"], b["hypothesis"], truncation="only_first", max_length=384),
        batched=True,
        remove_columns=["premise", "hypothesis"],
    )
    loader = DataLoader(test_ds, batch_size=16, collate_fn=DataCollatorWithPadding(tokenizer))

    os.makedirs("./demo_test_predictions", exist_ok=True)
    test_summary = []

    print("\n=================================================================")
    print("STEP 2: TESTING ALL 3 SAVED MODELS ON TEST.JSONL")
    print("=================================================================")

    for display_name, folder_name in variants:
        weights_path = os.path.join(DRIVE_DIR, folder_name, "best_model_weights.pt")
        model = DebertaV3EntailmentMSD(model_source, num_labels=2, offline=use_offline)
        model.load_state_dict(torch.load(weights_path, map_location=device))
        model.to(device)
        model.eval()

        logits_list = []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                logits_list.append(model(**batch).logits.cpu())

        full_logits = torch.cat(logits_list, dim=0)
        pred_ids = torch.argmax(full_logits, dim=-1).numpy()

        out_path = f"./demo_test_predictions/{folder_name}_test_predictions.jsonl"
        with open(out_path, "w") as f:
            for idx, p_id in zip(test_df["idx"], pred_ids):
                f.write(json.dumps({"idx": int(idx), "label": id2label[int(p_id)]}) + "\n")

        if has_test_labels:
            t_loss = F.cross_entropy(full_logits, y_test_tensor).item()
            t_acc = accuracy_score(y_test_true, pred_ids)
            t_f1 = f1_score(y_test_true, pred_ids, average="binary")
            test_summary.append({
                "Model Variant": display_name,
                "Test Loss": round(t_loss, 4),
                "Test Accuracy": round(t_acc, 4),
                "Test F1": round(t_f1, 4),
                "Predictions File": out_path,
            })
        else:
            test_summary.append({
                "Model Variant": display_name,
                "Test Loss": "Hidden",
                "Test Accuracy": "Hidden",
                "Test F1": "Hidden",
                "Predictions File": out_path,
            })

    test_res_df = pd.DataFrame(test_summary)
    print(test_res_df.to_string(index=False))

    if has_test_labels:
        best_model = test_res_df.sort_values(by="Test Accuracy", ascending=False).iloc
        print(f"\n>>> BEST MODEL: {best_model['Model Variant']} | Test Accuracy: {best_model['Test Accuracy']} | Test Loss: {best_model['Test Loss']} <<<")
else:
    print("\nNo test.jsonl uploaded yet — Step 1 Validation check completed cleanly!")


Click 'Choose Files' below to upload val.jsonl and/or test.jsonl (or a .zip):


Saving val.jsonl to val.jsonl
Mounted at /content/drive

STEP 1: SAVED VALIDATION RESULTS (BEFORE RUNNING TEST DATA)
                 Model Variant                    Saved Folder  Validation Accuracy  Validation F1
  Model 1 (seed=42, lr=2.0e-5)    submission_base_seed42_lr2e5               0.8520         0.8551
 Model 2 (seed=100, lr=1.5e-5)  submission_base_seed100_lr15e5               0.8339         0.8392
Model 3 (seed=2026, lr=2.5e-5) submission_base_seed2026_lr25e5               0.8412         0.8571

No test.jsonl uploaded yet — Step 1 Validation check completed cleanly!
